In [2]:
%cd /content
!git clone --branch feat/cuda-training-kernels --single-branch https://github.com/sagnik1511/wtf.git
%cd /content/wtf

!python --version
!nvidia-smi
!nvcc --version
!python -m pip install -e . setuptools wheel

/content
Cloning into 'wtf'...
remote: Enumerating objects: 3697, done.
remote: Counting objects: 100% (1090/1090), done.
remote: Compressing objects: 100% (757/757), done.
remote: Total 3697 (delta 657), reused 627 (delta 320), pack-reused 2607 (from 1)
Receiving objects: 100% (3697/3697), 24.53 MiB | 43.69 MiB/s, done.
Resolving deltas: 100% (2486/2486), done.
/content/wtf
Python 3.13.16
Wed Nov 25 12:09:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|======================

In [3]:
%cd /content/wtf
!git switch feat/cuda-training-kernels
!git pull --ff-only origin feat/cuda-training-kernels
!git log -1 --oneline
!python --version
!nvidia-smi
!nvcc --version

!python -m work.build_native
!python -m wtf.native.cuda_build
!WTF_TEST_CUDA=1 python -m unittest tests.test_cuda -v

/content/wtf
Already on 'feat/cuda-training-kernels'
Your branch is up to date with 'origin/feat/cuda-training-kernels'.
From https://github.com/sagnik1511/wtf
 * branch            feat/cuda-training-kernels -> FETCH_HEAD
Already up to date.
f8de32f (HEAD -> feat/cuda-training-kernels, origin/feat/cuda-training-kernels) test: cover CUDA KL rejection and record optimization validation
Python 3.13.16
Wed Nov 25 12:10:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=============

In [4]:
import os
import sys
import subprocess

common = [
    sys.executable, "-m", "work.benchmark_cuda",
    "--algorithms", "ppo", "dqn", "ddqn",
    "--steps", "1000",
    "--warmup", "32",
    "--hidden", "64", "64", "32",
    "--seed", "100",
    "--repeats", "1",
    "--cuda-inference", "auto",
]

cases = [
    ("fp64", [
        "--devices", "cpu", "cuda",
        "--cuda-precision", "fp64",
    ]),
    ("fp32", [
        "--devices", "cpu", "cuda",
        "--cuda-precision", "fp32",
    ]),
    ("profile-fp64", [
        "--devices", "cuda",
        "--cuda-precision", "fp64",
        "--cuda-profile",
    ]),
]

for name, options in cases:
    print(f"\nRunning {name}", flush=True)
    subprocess.run(
        common + options + [
            "--output", f".cache/cuda-{name}-report.json"
        ],
        env={**os.environ, "WTF_KERNELS": "native"},
        check=True,
    )


Running fp64


ppo cpu: 1000 steps, 14522 updates, 29.534s
ppo cuda: 1000 steps, 14522 updates, 19.860s
dqn cpu: 1000 steps, 8144 updates, 15.198s
dqn cuda: 1000 steps, 8144 updates, 9.565s
ddqn cpu: 1000 steps, 8714 updates, 19.970s



Running fp32


ddqn cuda: 1000 steps, 8714 updates, 10.657s
ppo cpu: 1000 steps, 14522 updates, 29.131s
ppo cuda: 1000 steps, 14522 updates, 16.849s
dqn cpu: 1000 steps, 8144 updates, 15.251s
dqn cuda: 1000 steps, 8144 updates, 13.103s
ddqn cpu: 1000 steps, 8714 updates, 20.280s



Running profile-fp64


ddqn cuda: 1000 steps, 8714 updates, 13.937s
ppo cuda: 1000 steps, 14522 updates, 20.431s
dqn cuda: 1000 steps, 8144 updates, 9.540s
ddqn cuda: 1000 steps, 8714 updates, 10.583s


In [5]:
import json
from pathlib import Path

fields = (
    "algorithm", "device", "world_seed", "steps", "updates",
    "initialization_seconds", "loop_seconds",
    "warmup_steps", "warmup_seconds", "steady_seconds",
    "cpu_backend", "accelerator",
)

for name in ("fp64", "fp32", "profile-fp64"):
    path = Path(f".cache/cuda-{name}-report.json")
    if not path.exists():
        print(f"\nMissing: {path}")
        continue

    report = json.loads(path.read_text())
    records = []
    for run in report["records"]:
        row = {key: run.get(key) for key in fields}
        row["peak_host_ram_mib"] = round(
            run["peak_rss_bytes"] / 2**20, 2
        )
        records.append(row)

    print(json.dumps({
        "case": name,
        "records": records,
        "comparisons": report["comparisons"],
        "failures": report["failures"],
    }, indent=2))

{
  "case": "fp64",
  "records": [
    {
      "algorithm": "ppo",
      "device": "cpu",
      "world_seed": 100,
      "steps": 1000,
      "updates": 14522,
      "initialization_seconds": 0.17520131699998842,
      "loop_seconds": 29.533758527999993,
      "warmup_steps": 32,
      "warmup_seconds": 0.3838118120000047,
      "steady_seconds": 29.149946715999988,
      "cpu_backend": {
        "backend": "native",
        "requested": "native",
        "fallback_reason": null,
        "native_api_version": 2
      },
      "accelerator": {},
      "peak_host_ram_mib": 51.03
    },
    {
      "algorithm": "ppo",
      "device": "cuda",
      "world_seed": 100,
      "steps": 1000,
      "updates": 14522,
      "initialization_seconds": 0.42981356899997536,
      "loop_seconds": 19.860400204,
      "warmup_steps": 32,
      "warmup_seconds": 0.5884702369999957,
      "steady_seconds": 19.271929967000005,
      "cpu_backend": {
        "backend": "native",
        "requested": "native

In [6]:
import json
import os
import subprocess
import sys
from pathlib import Path

cases = [
    ("fp32-profile", ["dqn", "ddqn"], "fp32", 256, True),
    ("cache-32m", ["ppo", "dqn", "ddqn"], "fp64", 32, False),
    ("cache-256m", ["ppo", "dqn", "ddqn"], "fp64", 256, False),
]

for name, algorithms, precision, memory, profile in cases:
    output = Path(f".cache/extra-{name}.json")
    command = [
        sys.executable, "-m", "work.benchmark_cuda",
        "--algorithms", *algorithms,
        "--devices", "cuda",
        "--steps", "1000",
        "--warmup", "32",
        "--hidden", "64", "64", "32",
        "--seed", "100",
        "--repeats", "1",
        "--cuda-inference", "auto",
        "--cuda-precision", precision,
        "--workspace-mb", str(memory),
        "--output", str(output),
    ]
    if profile:
        command.append("--cuda-profile")

    print(f"\nRUNNING: {name}", flush=True)
    process = subprocess.run(
        command,
        env={**os.environ, "WTF_KERNELS": "native"},
        check=False,
    )

    if output.exists():
        report = json.loads(output.read_text())
        print(json.dumps({
            "case": name,
            "records": [
                {
                    "algorithm": r["algorithm"],
                    "steps": r["steps"],
                    "updates": r["updates"],
                    "loop_seconds": r["loop_seconds"],
                    "peak_host_ram_mib": r["peak_rss_bytes"] / 2**20,
                    "accelerator": r["accelerator"],
                }
                for r in report["records"]
            ],
            "failures": report["failures"],
        }, indent=2))

    process.check_returncode()


RUNNING: fp32-profile


dqn cuda: 1000 steps, 8144 updates, 13.315s


{
  "case": "fp32-profile",
  "records": [
    {
      "algorithm": "dqn",
      "steps": 1000,
      "updates": 8144,
      "loop_seconds": 13.314951784000073,
      "peak_host_ram_mib": 373.54296875,
      "accelerator": {
        "precision": "fp32",
        "profile": true,
        "inference": "auto",
        "packing_seconds": 1.414840721999326,
        "cpu_optimizer_seconds": 7.972348882999427,
        "cuda_upload_seconds": 0.23125529594719407,
        "cuda_compute_seconds": 0.13415036800503732,
        "cuda_download_seconds": 0.05387452796101571,
        "host_to_device_bytes": 327593824,
        "device_to_host_bytes": 253539008,
        "weight_uploads": 290,
        "weight_cache_hits": 196,
        "ppo_batches": 0,
        "ppo_model_updates": 0,
        "backend": "cuda-experimental",
        "device": "Tesla T4",
        "workspace_bytes": 175940216,
        "workspace_budget_bytes": 268435456,
        "cached_workspaces": 41,
        "batches": 243,
        "model_u

ddqn cuda: 1000 steps, 8714 updates, 14.138s
ppo cuda: 1000 steps, 14522 updates, 20.141s
dqn cuda: 1000 steps, 8144 updates, 9.516s


{
  "case": "cache-32m",
  "records": [
    {
      "algorithm": "ppo",
      "steps": 1000,
      "updates": 14522,
      "loop_seconds": 20.14051208500007,
      "peak_host_ram_mib": 173.0078125,
      "accelerator": {
        "precision": "fp64",
        "profile": false,
        "inference": "auto",
        "packing_seconds": 0.0,
        "cpu_optimizer_seconds": 0.0,
        "cuda_upload_seconds": 0.0,
        "cuda_compute_seconds": 0.0,
        "cuda_download_seconds": 0.0,
        "host_to_device_bytes": 282951800,
        "device_to_host_bytes": 130020984,
        "weight_uploads": 155,
        "weight_cache_hits": 0,
        "ppo_batches": 2260,
        "ppo_model_updates": 937,
        "backend": "cuda-experimental",
        "device": "Tesla T4",
        "workspace_bytes": 27468224,
        "workspace_budget_bytes": 33554432,
        "cached_workspaces": 5,
        "batches": 0,
        "model_updates": 0,
        "prediction_batches": 0,
        "predictions": 0
      }
   

ddqn cuda: 1000 steps, 8714 updates, 10.625s
ppo cuda: 1000 steps, 14522 updates, 20.116s
dqn cuda: 1000 steps, 8144 updates, 9.640s


{
  "case": "cache-256m",
  "records": [
    {
      "algorithm": "ppo",
      "steps": 1000,
      "updates": 14522,
      "loop_seconds": 20.115668592999896,
      "peak_host_ram_mib": 259.27734375,
      "accelerator": {
        "precision": "fp64",
        "profile": false,
        "inference": "auto",
        "packing_seconds": 0.0,
        "cpu_optimizer_seconds": 0.0,
        "cuda_upload_seconds": 0.0,
        "cuda_compute_seconds": 0.0,
        "cuda_download_seconds": 0.0,
        "host_to_device_bytes": 282951672,
        "device_to_host_bytes": 130020984,
        "weight_uploads": 155,
        "weight_cache_hits": 0,
        "ppo_batches": 2260,
        "ppo_model_updates": 937,
        "backend": "cuda-experimental",
        "device": "Tesla T4",
        "workspace_bytes": 174987576,
        "workspace_budget_bytes": 268435456,
        "cached_workspaces": 19,
        "batches": 0,
        "model_updates": 0,
        "prediction_batches": 0,
        "predictions": 0
     

ddqn cuda: 1000 steps, 8714 updates, 10.741s


In [7]:
import json
import unittest

from tests.test_advanced_algorithms import environment
from wtf.experiment.runner import Experiment


class GPUResumeChecks(unittest.TestCase):
    def test_checkpoint_resume(self):
        for kind in ("dqn", "ddqn", "ppo", "maskable_ppo"):
            for precision in ("fp64", "fp32"):
                for inference in ("auto", "cuda"):
                    with self.subTest(
                        algorithm=kind,
                        precision=precision,
                        inference=inference,
                    ):
                        config = environment(kind)
                        config["experiment"]["runs"] = 1
                        options = dict(
                            precision=precision,
                            inference=inference,
                            max_agents=3,
                            workspace_mb=8,
                        )

                        original = Experiment(config)
                        restored = None
                        try:
                            original.set_training_device("cuda", **options)
                            for _ in range(5):
                                original.step()

                            self.assertGreater(original.total_updates, 0)

                            state = original.checkpoint()
                            transported = json.loads(
                                json.dumps(state, allow_nan=False)
                            )
                            restored = Experiment.restore(transported)

                            self.assertIsNone(restored._cuda_training)
                            self.assertEqual(state, restored.checkpoint())

                            restored.set_training_device("cuda", **options)

                            for _ in range(3):
                                original.step()
                                restored.step()
                                self.assertEqual(
                                    original.checkpoint(),
                                    restored.checkpoint(),
                                )
                        finally:
                            original.set_training_device("cpu")
                            if restored is not None:
                                restored.set_training_device("cpu")


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(GPUResumeChecks)
)
assert result.wasSuccessful(), "GPU checkpoint/resume check failed"

test_checkpoint_resume (__main__.GPUResumeChecks.test_checkpoint_resume) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.927s

OK


In [8]:
import os
import shutil
import subprocess
import sys

sanitizer = shutil.which("compute-sanitizer")

if sanitizer is None:
    print("SKIPPED: compute-sanitizer is unavailable in this runtime.")
else:
    for tool in ("memcheck", "initcheck", "racecheck", "synccheck"):
        print(f"\nRUNNING COMPUTE SANITIZER: {tool}", flush=True)
        subprocess.run(
            [
                sanitizer,
                "--tool", tool,
                "--error-exitcode", "99",
                sys.executable,
                "-m", "unittest",
                "tests.test_cuda.CudaHardwareTests",
                "-v",
            ],
            env={**os.environ, "WTF_TEST_CUDA": "1"},
            check=True,
        )


RUNNING COMPUTE SANITIZER: memcheck
========= COMPUTE-SANITIZER


test_real_fp32_forward_gradient_cache_and_timing (tests.test_cuda.CudaHardwareTests.test_real_fp32_forward_gradient_cache_and_timing) ... ok
test_real_gradients_and_dense_forwards_agree_with_reference (tests.test_cuda.CudaHardwareTests.test_real_gradients_and_dense_forwards_agree_with_reference) ... ok
test_real_ppo_kl_rejection_is_per_animal (tests.test_cuda.CudaHardwareTests.test_real_ppo_kl_rejection_is_per_animal) ... ok
test_real_resident_ppo_and_fp32 (tests.test_cuda.CudaHardwareTests.test_real_resident_ppo_and_fp32) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.480s

OK


========= ERROR SUMMARY: 0 errors

RUNNING COMPUTE SANITIZER: initcheck
========= COMPUTE-SANITIZER


test_real_fp32_forward_gradient_cache_and_timing (tests.test_cuda.CudaHardwareTests.test_real_fp32_forward_gradient_cache_and_timing) ... ok
test_real_gradients_and_dense_forwards_agree_with_reference (tests.test_cuda.CudaHardwareTests.test_real_gradients_and_dense_forwards_agree_with_reference) ... ok
test_real_ppo_kl_rejection_is_per_animal (tests.test_cuda.CudaHardwareTests.test_real_ppo_kl_rejection_is_per_animal) ... ok
test_real_resident_ppo_and_fp32 (tests.test_cuda.CudaHardwareTests.test_real_resident_ppo_and_fp32) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.523s

OK


========= ERROR SUMMARY: 0 errors

RUNNING COMPUTE SANITIZER: racecheck
========= COMPUTE-SANITIZER


test_real_fp32_forward_gradient_cache_and_timing (tests.test_cuda.CudaHardwareTests.test_real_fp32_forward_gradient_cache_and_timing) ... ok
test_real_gradients_and_dense_forwards_agree_with_reference (tests.test_cuda.CudaHardwareTests.test_real_gradients_and_dense_forwards_agree_with_reference) ... ok
test_real_ppo_kl_rejection_is_per_animal (tests.test_cuda.CudaHardwareTests.test_real_ppo_kl_rejection_is_per_animal) ... ok
test_real_resident_ppo_and_fp32 (tests.test_cuda.CudaHardwareTests.test_real_resident_ppo_and_fp32) ... ok

----------------------------------------------------------------------
Ran 4 tests in 1.042s

OK


========= RACECHECK SUMMARY: 0 hazards displayed (0 errors, 0 warnings)

RUNNING COMPUTE SANITIZER: synccheck
========= COMPUTE-SANITIZER


test_real_fp32_forward_gradient_cache_and_timing (tests.test_cuda.CudaHardwareTests.test_real_fp32_forward_gradient_cache_and_timing) ... ok
test_real_gradients_and_dense_forwards_agree_with_reference (tests.test_cuda.CudaHardwareTests.test_real_gradients_and_dense_forwards_agree_with_reference) ... ok
test_real_ppo_kl_rejection_is_per_animal (tests.test_cuda.CudaHardwareTests.test_real_ppo_kl_rejection_is_per_animal) ... ok
test_real_resident_ppo_and_fp32 (tests.test_cuda.CudaHardwareTests.test_real_resident_ppo_and_fp32) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.421s

OK


========= ERROR SUMMARY: 0 errors


In [9]:
%cd /content/wtf

import math
import random
import unittest
from array import array

from wtf.learning.cuda_training import CudaTraining
from wtf.learning.ppo import MLP


class ExtraGPUChecks(unittest.TestCase):
    def check_predictions(self, engine, jobs):
        actual = engine.predict_many(
            [(net.layers, obs) for net, obs in jobs]
        )
        self.assertEqual(len(actual), len(jobs))

        fp32 = engine.precision == "fp32"
        for (net, obs), values in zip(jobs, actual):
            inputs = tuple(array("f", obs)) if fp32 else obs
            expected = net.predict(inputs)
            self.assertEqual(len(values), len(expected))

            for got, want in zip(values, expected):
                self.assertTrue(math.isfinite(got))
                self.assertTrue(
                    math.isclose(
                        got,
                        want,
                        rel_tol=1e-4 if fp32 else 1e-9,
                        abs_tol=2e-5 if fp32 else 1e-10,
                    ),
                    f"{engine.precision}: {got=} {want=}",
                )

    def test_mixed_shapes_and_weight_changes(self):
        rng = random.Random(91)
        shapes = [
            (1, [], 1),
            (17, [], 3),
            (33, [65, 17], 7),
            (257, [63, 129], 13),
            (7, [3] * 15, 2),
        ]

        for precision in ("fp64", "fp32"):
            with self.subTest(precision=precision):
                engine = CudaTraining(
                    precision=precision,
                    max_agents=4,
                    workspace_mb=8,
                )
                try:
                    jobs = []
                    for inputs, hidden, outputs in shapes:
                        for index in range(7):
                            net = MLP(
                                hidden,
                                outputs,
                                100 + index,
                                value_head=False,
                                inputs=inputs,
                            )
                            obs = tuple(
                                rng.uniform(-1, 1)
                                for _ in range(inputs)
                            )
                            jobs.append((net, obs))

                    rng.shuffle(jobs)

                    self.check_predictions(engine, jobs)
                    self.check_predictions(engine, jobs)  # Cached weights.

                    # Same arrays, changed contents: cache must notice.
                    for net, _ in jobs:
                        net.layers[-1]["bias"][0] += 0.125

                    self.check_predictions(engine, jobs)
                    self.check_predictions(engine, list(reversed(jobs)))
                finally:
                    engine.close()

    def test_workspace_eviction_and_reuse(self):
        rng = random.Random(103)
        jobs = [
            (
                MLP(
                    [160, 160],
                    5,
                    20 + index,
                    value_head=False,
                    inputs=33,
                ),
                tuple(rng.uniform(-1, 1) for _ in range(33)),
            )
            for index in range(4)
        ]

        for precision in ("fp64", "fp32"):
            with self.subTest(precision=precision):
                engine = CudaTraining(
                    precision=precision,
                    max_agents=4,
                    workspace_mb=1,
                )
                try:
                    for count in (1, 2, 3, 4, 1, 4, 2):
                        self.check_predictions(engine, jobs[:count])
                        self.assertLessEqual(
                            engine.allocated, 1024 * 1024
                        )

                    # All four workspace sizes cannot remain cached.
                    self.assertLess(len(engine.workspaces), 4)
                finally:
                    engine.close()

                self.assertEqual(engine.allocated, 0)
                self.assertFalse(engine.workspaces)
                self.assertFalse(engine.buffers)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(ExtraGPUChecks)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Additional GPU correctness checks failed"

test_mixed_shapes_and_weight_changes (__main__.ExtraGPUChecks.test_mixed_shapes_and_weight_changes) ... ok
test_workspace_eviction_and_reuse (__main__.ExtraGPUChecks.test_workspace_eviction_and_reuse) ... 

/content/wtf


ok

----------------------------------------------------------------------
Ran 2 tests in 0.203s

OK


In [10]:
import json
import os
from pathlib import Path
import subprocess
import sys

for seed in (101, 102, 103):
    output = Path(f".cache/cuda-seed-{seed}.json")
    output.parent.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            sys.executable, "-m", "work.benchmark_cuda",
            "--algorithms", "ppo", "dqn", "ddqn",
            "--devices", "cpu", "cuda",
            "--steps", "1000",
            "--warmup", "32",
            "--hidden", "64", "64", "32",
            "--seed", str(seed),
            "--repeats", "1",
            "--cuda-precision", "fp64",
            "--cuda-inference", "auto",
            "--output", str(output),
        ],
        env={**os.environ, "WTF_KERNELS": "native"},
        check=True,
    )

    report = json.loads(output.read_text())
    print(json.dumps({
        "seed": seed,
        "comparisons": report.get("comparisons"),
        "failures": report.get("failures"),
    }, indent=2))

ppo cpu: 1000 steps, 11073 updates, 23.119s
ppo cuda: 1000 steps, 11073 updates, 15.992s
dqn cpu: 1000 steps, 8018 updates, 14.926s
dqn cuda: 1000 steps, 8018 updates, 9.517s
ddqn cpu: 1000 steps, 7836 updates, 18.393s
ddqn cuda: 1000 steps, 7836 updates, 9.773s


{
  "seed": 101,
  "comparisons": [
    {
      "algorithm": "ppo",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.4456214389380369
    },
    {
      "algorithm": "dqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.5683505545777914
    },
    {
      "algorithm": "ddqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.88195556558197
    }
  ],
  "failures": []
}


ppo cpu: 1000 steps, 12029 updates, 25.113s
ppo cuda: 1000 steps, 12029 updates, 16.495s
dqn cpu: 1000 steps, 10124 updates, 18.622s
dqn cuda: 1000 steps, 10124 updates, 11.773s
ddqn cpu: 1000 steps, 9302 updates, 21.312s
ddqn cuda: 1000 steps, 9302 updates, 11.381s


{
  "seed": 102,
  "comparisons": [
    {
      "algorithm": "ppo",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.522487585442069
    },
    {
      "algorithm": "dqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.5817351079611788
    },
    {
      "algorithm": "ddqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.8726229717207037
    }
  ],
  "failures": []
}


ppo cpu: 1000 steps, 13983 updates, 28.365s
ppo cuda: 1000 steps, 13983 updates, 18.938s
dqn cpu: 1000 steps, 8369 updates, 15.392s
dqn cuda: 1000 steps, 8369 updates, 9.843s
ddqn cpu: 1000 steps, 8736 updates, 19.832s


{
  "seed": 103,
  "comparisons": [
    {
      "algorithm": "ppo",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.4977760901588633
    },
    {
      "algorithm": "dqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.5636345411586747
    },
    {
      "algorithm": "ddqn",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.8729706244519742
    }
  ],
  "failures": []
}


ddqn cuda: 1000 steps, 8736 updates, 10.589s


In [11]:
!WTF_KERNELS=native python -m work.benchmark_cuda \
  --algorithms maskable_ppo a2c reinforce grpo mappo qrdqn trpo discrete_sac \
  --devices cpu cuda \
  --steps 300 \
  --warmup 32 \
  --hidden 64 64 32 \
  --seed 100 \
  --repeats 1 \
  --cuda-precision fp64 \
  --cuda-inference auto \
  --output .cache/cuda-other-policies.json

import json
from pathlib import Path

report = json.loads(
    Path(".cache/cuda-other-policies.json").read_text()
)
print(json.dumps({
    "comparisons": report.get("comparisons"),
    "failures": report.get("failures"),
}, indent=2))

maskable_ppo cpu: 300 steps, 3916 updates, 8.076s
maskable_ppo cuda: 300 steps, 3916 updates, 5.339s
a2c cpu: 300 steps, 557 updates, 3.370s
a2c cuda: 300 steps, 557 updates, 3.638s
reinforce cpu: 300 steps, 61 updates, 2.423s
reinforce cuda: 300 steps, 61 updates, 2.629s
grpo cpu: 300 steps, 178 updates, 6.380s
grpo cuda: 300 steps, 178 updates, 6.658s
mappo cpu: 300 steps, 3952 updates, 9.368s
mappo cuda: 300 steps, 3952 updates, 9.852s
qrdqn cpu: 300 steps, 3831 updates, 15.409s
qrdqn cuda: 300 steps, 3831 updates, 15.892s
trpo cpu: 300 steps, 2366 updates, 467.018s
trpo cuda: 300 steps, 2366 updates, 470.230s
discrete_sac cpu: 300 steps, 17586 updates, 326.433s
discrete_sac cuda: 300 steps, 17586 updates, 333.801s
{
  "comparisons": [
    {
      "algorithm": "maskable_ppo",
      "matched_work_counts": true,
      "cuda_exercised": true,
      "profiled": false,
      "loop_speedup": 1.5127375573899937
    },
    {
      "algorithm": "a2c",
      "matched_work_counts": true,
     

In [13]:
%cd /content/wtf

!WTF_KERNELS=native python -m work.benchmark_cuda \
  --algorithms ppo dqn ddqn \
  --devices cpu cuda \
  --steps 1000 --warmup 32 \
  --hidden 64 64 32 \
  --seed 100 --repeats 3 \
  --cuda-precision fp64 \
  --cuda-inference auto \
  --output .cache/perf-baseline.json

/content/wtf
ppo cpu: 1000 steps, 14522 updates, 29.131s
ppo cuda: 1000 steps, 14522 updates, 20.202s
ppo cpu: 1000 steps, 11073 updates, 23.297s
ppo cuda: 1000 steps, 11073 updates, 16.246s
ppo cpu: 1000 steps, 12029 updates, 25.070s
ppo cuda: 1000 steps, 12029 updates, 16.822s
dqn cpu: 1000 steps, 8144 updates, 15.575s
dqn cuda: 1000 steps, 8144 updates, 9.733s
dqn cpu: 1000 steps, 8018 updates, 15.009s
dqn cuda: 1000 steps, 8018 updates, 9.488s
dqn cpu: 1000 steps, 10124 updates, 18.659s
dqn cuda: 1000 steps, 10124 updates, 11.681s
ddqn cpu: 1000 steps, 8714 updates, 20.435s
ddqn cuda: 1000 steps, 8714 updates, 10.892s
ddqn cpu: 1000 steps, 7836 updates, 18.701s
ddqn cuda: 1000 steps, 7836 updates, 10.127s
ddqn cpu: 1000 steps, 9302 updates, 23.234s
ddqn cuda: 1000 steps, 9302 updates, 12.140s


In [14]:
import json
import os
from pathlib import Path
import statistics
import subprocess
import sys

def summarize(path):
    report = json.loads(Path(path).read_text())

    for algorithm in sorted({r["algorithm"] for r in report["records"]}):
        for device in ("cpu", "cuda"):
            rows = [
                r for r in report["records"]
                if r["algorithm"] == algorithm and r["device"] == device
            ]
            if not rows:
                continue

            print(json.dumps({
                "algorithm": algorithm,
                "device": device,
                "median_loop_s": round(statistics.median(
                    r["loop_seconds"] for r in rows
                ), 3),
                "median_steps_per_s": round(statistics.median(
                    r["steps"] / r["loop_seconds"] for r in rows
                ), 2),
                "max_host_peak_MiB": round(max(
                    r["peak_rss_bytes"] for r in rows
                ) / 2**20, 2),
                "max_retained_GPU_workspace_MiB": round(max(
                    r.get("accelerator", {}).get("workspace_bytes", 0)
                    for r in rows
                ) / 2**20, 2),
                "steps": [r["steps"] for r in rows],
                "updates": [r["updates"] for r in rows],
            }))

    print("Comparisons:", report["comparisons"])
    print("Failures:", report["failures"])


summarize(".cache/perf-baseline.json")

# Baseline above already measures batch=64, workspace=256 MiB.
cases = [
    ("batch8", 8, 256),
    ("batch32", 32, 256),
    ("workspace32", 64, 32),
    ("workspace64", 64, 64),
]

for name, batch, workspace in cases:
    output = f".cache/perf-{name}.json"
    subprocess.run(
        [
            sys.executable, "-m", "work.benchmark_cuda",
            "--algorithms", "ppo", "dqn", "ddqn",
            "--devices", "cuda",
            "--steps", "1000", "--warmup", "32",
            "--hidden", "64", "64", "32",
            "--seed", "100", "--repeats", "3",
            "--cuda-precision", "fp64",
            "--cuda-inference", "auto",
            "--batch-agents", str(batch),
            "--workspace-mb", str(workspace),
            "--output", output,
        ],
        env={**os.environ, "WTF_KERNELS": "native"},
        check=True,
    )
    print(f"\nCASE: {name}")
    summarize(output)

{"algorithm": "ddqn", "device": "cpu", "median_loop_s": 20.435, "median_steps_per_s": 48.94, "max_host_peak_MiB": 51.89, "max_retained_GPU_workspace_MiB": 0.0, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 10.892, "median_steps_per_s": 91.81, "max_host_peak_MiB": 395.68, "max_retained_GPU_workspace_MiB": 229.05, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "dqn", "device": "cpu", "median_loop_s": 15.575, "median_steps_per_s": 64.21, "max_host_peak_MiB": 52.89, "max_retained_GPU_workspace_MiB": 0.0, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 9.733, "median_steps_per_s": 102.74, "max_host_peak_MiB": 416.81, "max_retained_GPU_workspace_MiB": 248.36, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "ppo", "device": "cpu", "median_loop_s": 25.07, "median_steps_per_s": 39.89, "max_host_peak_MiB"

ppo cuda: 1000 steps, 14522 updates, 21.628s
ppo cuda: 1000 steps, 11073 updates, 16.524s
ppo cuda: 1000 steps, 12029 updates, 18.204s
dqn cuda: 1000 steps, 8144 updates, 10.225s
dqn cuda: 1000 steps, 8018 updates, 10.342s
dqn cuda: 1000 steps, 10124 updates, 12.876s
ddqn cuda: 1000 steps, 8714 updates, 11.643s
ddqn cuda: 1000 steps, 7836 updates, 10.391s
ddqn cuda: 1000 steps, 9302 updates, 12.124s



CASE: batch8
{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 11.643, "median_steps_per_s": 85.89, "max_host_peak_MiB": 160.48, "max_retained_GPU_workspace_MiB": 6.1, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 10.342, "median_steps_per_s": 96.69, "max_host_peak_MiB": 161.57, "max_retained_GPU_workspace_MiB": 6.1, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "ppo", "device": "cuda", "median_loop_s": 18.204, "median_steps_per_s": 54.93, "max_host_peak_MiB": 164.72, "max_retained_GPU_workspace_MiB": 18.64, "steps": [1000, 1000, 1000], "updates": [14522, 11073, 12029]}
Comparisons: [{'algorithm': 'ppo', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'dqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'ddqn', 'matched_work_counts': False, 'cuda_exercised'

ppo cuda: 1000 steps, 14522 updates, 20.579s
ppo cuda: 1000 steps, 11073 updates, 16.755s
ppo cuda: 1000 steps, 12029 updates, 17.625s
dqn cuda: 1000 steps, 8144 updates, 9.553s
dqn cuda: 1000 steps, 8018 updates, 9.431s
dqn cuda: 1000 steps, 10124 updates, 11.695s
ddqn cuda: 1000 steps, 8714 updates, 10.522s
ddqn cuda: 1000 steps, 7836 updates, 9.657s
ddqn cuda: 1000 steps, 9302 updates, 11.335s



CASE: batch32
{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 10.522, "median_steps_per_s": 95.04, "max_host_peak_MiB": 240.07, "max_retained_GPU_workspace_MiB": 82.5, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 9.553, "median_steps_per_s": 104.68, "max_host_peak_MiB": 241.01, "max_retained_GPU_workspace_MiB": 83.18, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "ppo", "device": "cuda", "median_loop_s": 17.625, "median_steps_per_s": 56.74, "max_host_peak_MiB": 209.05, "max_retained_GPU_workspace_MiB": 87.76, "steps": [1000, 1000, 1000], "updates": [14522, 11073, 12029]}
Comparisons: [{'algorithm': 'ppo', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'dqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'ddqn', 'matched_work_counts': False, 'cuda_exerci

ppo cuda: 1000 steps, 14522 updates, 20.079s
ppo cuda: 1000 steps, 11073 updates, 15.790s
ppo cuda: 1000 steps, 12029 updates, 16.657s
dqn cuda: 1000 steps, 8144 updates, 9.472s
dqn cuda: 1000 steps, 8018 updates, 9.287s
dqn cuda: 1000 steps, 10124 updates, 11.667s
ddqn cuda: 1000 steps, 8714 updates, 10.541s
ddqn cuda: 1000 steps, 7836 updates, 9.491s
ddqn cuda: 1000 steps, 9302 updates, 11.339s



CASE: workspace32
{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 10.541, "median_steps_per_s": 94.87, "max_host_peak_MiB": 193.02, "max_retained_GPU_workspace_MiB": 31.0, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 9.472, "median_steps_per_s": 105.58, "max_host_peak_MiB": 195.05, "max_retained_GPU_workspace_MiB": 31.85, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "ppo", "device": "cuda", "median_loop_s": 16.657, "median_steps_per_s": 60.03, "max_host_peak_MiB": 175.07, "max_retained_GPU_workspace_MiB": 31.37, "steps": [1000, 1000, 1000], "updates": [14522, 11073, 12029]}
Comparisons: [{'algorithm': 'ppo', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'dqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'ddqn', 'matched_work_counts': False, 'cuda_ex

ppo cuda: 1000 steps, 14522 updates, 20.154s
ppo cuda: 1000 steps, 11073 updates, 16.002s
ppo cuda: 1000 steps, 12029 updates, 16.745s
dqn cuda: 1000 steps, 8144 updates, 9.749s
dqn cuda: 1000 steps, 8018 updates, 9.484s
dqn cuda: 1000 steps, 10124 updates, 11.847s
ddqn cuda: 1000 steps, 8714 updates, 10.902s
ddqn cuda: 1000 steps, 7836 updates, 9.583s



CASE: workspace64
{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 10.902, "median_steps_per_s": 91.72, "max_host_peak_MiB": 227.6, "max_retained_GPU_workspace_MiB": 62.01, "steps": [1000, 1000, 1000], "updates": [8714, 7836, 9302]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 9.749, "median_steps_per_s": 102.57, "max_host_peak_MiB": 227.05, "max_retained_GPU_workspace_MiB": 63.36, "steps": [1000, 1000, 1000], "updates": [8144, 8018, 10124]}
{"algorithm": "ppo", "device": "cuda", "median_loop_s": 16.745, "median_steps_per_s": 59.72, "max_host_peak_MiB": 194.8, "max_retained_GPU_workspace_MiB": 60.44, "steps": [1000, 1000, 1000], "updates": [14522, 11073, 12029]}
Comparisons: [{'algorithm': 'ppo', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'dqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'ddqn', 'matched_work_counts': False, 'cuda_exe

ddqn cuda: 1000 steps, 9302 updates, 11.580s


In [15]:
import threading
import time

gpu_samples = []
stop_sampling = threading.Event()

def sample_gpu():
    while not stop_sampling.is_set():
        sample = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,utilization.gpu",
                "--format=csv,noheader,nounits",
                "--id=0",
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        if sample.returncode == 0:
            try:
                memory, utilization = map(
                    float, sample.stdout.strip().split(",")
                )
                gpu_samples.append((time.monotonic(), memory, utilization))
            except ValueError:
                pass
        stop_sampling.wait(0.5)

monitor = threading.Thread(target=sample_gpu, daemon=True)
monitor.start()

try:
    subprocess.run(
        [
            sys.executable, "-m", "work.benchmark_cuda",
            "--algorithms", "ppo", "dqn", "ddqn",
            "--devices", "cuda",
            "--steps", "3000", "--warmup", "64",
            "--hidden", "64", "64", "32",
            "--seed", "100", "--repeats", "1",
            "--cuda-precision", "fp64",
            "--cuda-inference", "auto",
            "--output", ".cache/perf-memory-long.json",
        ],
        env={**os.environ, "WTF_KERNELS": "native"},
        check=True,
    )
finally:
    stop_sampling.set()
    monitor.join()

summarize(".cache/perf-memory-long.json")

if gpu_samples:
    print({
        "sampled_peak_total_GPU_MiB": max(s[1] for s in gpu_samples),
        "median_sampled_GPU_utilization_pct": statistics.median(
            s[2] for s in gpu_samples
        ),
        "samples": len(gpu_samples),
    })

ppo cuda: 3000 steps, 47652 updates, 74.015s
dqn cuda: 3000 steps, 10199 updates, 12.993s


{"algorithm": "ddqn", "device": "cuda", "median_loop_s": 13.475, "median_steps_per_s": 173.22, "max_host_peak_MiB": 406.25, "max_retained_GPU_workspace_MiB": 240.06, "steps": [2334], "updates": [10312]}
{"algorithm": "dqn", "device": "cuda", "median_loop_s": 12.993, "median_steps_per_s": 230.9, "max_host_peak_MiB": 423.91, "max_retained_GPU_workspace_MiB": 251.92, "steps": [3000], "updates": [10199]}
{"algorithm": "ppo", "device": "cuda", "median_loop_s": 74.015, "median_steps_per_s": 40.53, "max_host_peak_MiB": 305.7, "max_retained_GPU_workspace_MiB": 230.46, "steps": [3000], "updates": [47652]}
Comparisons: [{'algorithm': 'ppo', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'dqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}, {'algorithm': 'ddqn', 'matched_work_counts': False, 'cuda_exercised': True, 'profiled': False, 'loop_speedup': None}]
Failures: []
{'sampled_peak_to

ddqn cuda: 2334 steps, 10312 updates, 13.475s


In [ ]:
# Run this single cell after building the CPU and CUDA kernels.
import json
import os
from pathlib import Path
import statistics
import subprocess
import sys
import tempfile
import time

# ---------- Configuration ----------
FULL = True                     # False: shorter first pass.
SEEDS = [100, 101, 102] if FULL else [100]
STEPS = 1000
CORE = ["ppo", "dqn", "ddqn"]

# Locate the existing checkout without assuming its location.
roots = [Path.cwd(), Path("/content/wtf"), Path("/kaggle/working/wtf")]
REPO = next(
    (p for p in roots if (p / "work/benchmark_cuda.py").exists()),
    None,
)
if REPO is None:
    raise RuntimeError(
        "Cannot find the checkout. Set REPO to its directory, "
        "then rerun this cell."
    )

ENV = {**os.environ, "WTF_KERNELS": "native"}

print(f"Repository: {REPO}", flush=True)
print(f"Python: {sys.version.split()[0]}", flush=True)
subprocess.run(["nvidia-smi"], check=False)

# ---------- Cases ----------
cases = []

def add(name, algorithms=CORE, devices=("cuda",),
        hidden=(64, 64, 32), precision="fp64", inference="auto",
        batch=64, workspace=256, profile=False, steps=STEPS,
        seeds=None):
    cases.append(dict(
        name=name, algorithms=algorithms, devices=devices,
        hidden=hidden, precision=precision, inference=inference,
        batch=batch, workspace=workspace, profile=profile,
        steps=steps, seeds=SEEDS if seeds is None else seeds,
    ))

add("baseline", devices=("cpu", "cuda"))
add("fp32", precision="fp32")
add("small", devices=("cpu", "cuda"), hidden=(32, 32))
add("large", devices=("cpu", "cuda"), hidden=(128, 128, 64))
add("large-gpu-inference", hidden=(128, 128, 64), inference="cuda")
add("batch8", batch=8)
add("workspace32", workspace=32)
add("profile-fp64", profile=True, seeds=[100])
add("profile-fp32", precision="fp32", profile=True, seeds=[100])

if FULL:
    add("batch32", batch=32)
    add("workspace64", workspace=64)
    add("long", steps=3000, seeds=[100])
    add(
        "other",
        algorithms=[
            "maskable_ppo", "a2c", "reinforce", "grpo",
            "mappo", "qrdqn", "trpo", "discrete_sac",
        ],
        devices=("cpu", "cuda"),
        steps=300,
        seeds=[100],
    )

jobs = [
    (case, algorithm, seed, device)
    for case in cases
    for algorithm in case["algorithms"]
    for seed in case["seeds"]
    for device in case["devices"]
]
print(f"\nRunning {len(jobs)} fresh-process trials sequentially.", flush=True)
print("No playback/checkpoint files or report JSON files are written.", flush=True)

# Results stay in notebook memory; every trial prints immediately.
results = []
errors = []

def sample_gpu():
    try:
        p = subprocess.run(
            [
                "nvidia-smi", "--id=0",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True, text=True, timeout=5,
        )
        return float(p.stdout.strip()) if p.returncode == 0 else None
    except (OSError, ValueError, subprocess.TimeoutExpired):
        return None

def med(values):
    return statistics.median(values)

suite_start = time.perf_counter()

# ---------- Execute ----------
try:
    for index, (case, algorithm, seed, device) in enumerate(jobs, 1):
        label = f'{case["name"]} / {algorithm} / {device} / seed {seed}'
        print(f"\n[{index}/{len(jobs)}] START {label}", flush=True)

        cmd = [
            sys.executable, "-m", "work.benchmark_cuda",
            "--worker", device,
            "--algorithm", algorithm,
            "--steps", str(case["steps"]),
            "--warmup", "32",
            "--hidden", *map(str, case["hidden"]),
            "--seed", str(seed),
            "--cuda-precision", case["precision"],
            "--cuda-inference", case["inference"],
            "--batch-agents", str(case["batch"]),
            "--workspace-mb", str(case["workspace"]),
        ]
        if case["profile"]:
            cmd.append("--cuda-profile")

        samples = []
        process = None

        # Temporary files prevent stdout/stderr pipe blockage.
        with tempfile.TemporaryFile(mode="w+t") as out, \
             tempfile.TemporaryFile(mode="w+t") as err:
            try:
                process = subprocess.Popen(
                    cmd, cwd=REPO, env=ENV, stdout=out, stderr=err
                )
                while process.poll() is None:
                    if device == "cuda":
                        value = sample_gpu()
                        if value is not None:
                            samples.append(value)
                    try:
                        process.wait(timeout=1)
                    except subprocess.TimeoutExpired:
                        pass
            finally:
                if process is not None and process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()

            out.seek(0)
            err.seek(0)
            text, stderr = out.read(), err.read()

        if process.returncode != 0:
            message = stderr or text or "No error output."
            errors.append((label, message))
            print(f"  FAILED:\n{message}", flush=True)
            continue

        try:
            r = json.loads(text)
        except json.JSONDecodeError:
            errors.append((label, text))
            print(f"  Invalid worker output:\n{text}\n{stderr}", flush=True)
            continue

        r.update(
            case=case["name"], seed=seed, profiled=case["profile"],
            requested_steps=case["steps"],
            gpu_peak=max(samples) if samples else None,
        )
        results.append(r)

        duration = r["loop_seconds"]
        print(
            f"  Loop: {duration:.3f} seconds\n"
            f'  Completed: {r["steps"]} steps, {r["updates"]} updates\n'
            f'  Speed: {r["steps"]/duration:.1f} steps/s; '
            f'{r["updates"]/duration:.1f} updates/s\n'
            f'  Peak host RAM: {r["peak_rss_bytes"]/2**20:.1f} MiB',
            flush=True,
        )
        if r["gpu_peak"] is not None:
            print(
                f'  Sampled total GPU memory peak: {r["gpu_peak"]:.1f} MiB',
                flush=True,
            )

except KeyboardInterrupt:
    print("\nStopped. Completed trials remain available below.", flush=True)

# ---------- Aggregate results ----------
print("\n" + "=" * 76)
print("SUMMARY")
print("=" * 76)

groups = {}
for r in results:
    groups.setdefault((r["case"], r["algorithm"], r["device"]), []).append(r)

for (case, algorithm, device), rows in sorted(groups.items()):
    print(
        f"\n{case} | {algorithm.upper()} | {device.upper()} | {len(rows)} trials\n"
        f'  Median loop: {med([r["loop_seconds"] for r in rows]):.3f} s\n'
        f'  Median updates/s: '
        f'{med([r["updates"]/r["loop_seconds"] for r in rows]):.1f}\n'
        f'  Highest host RAM peak: '
        f'{max(r["peak_rss_bytes"] for r in rows)/2**20:.1f} MiB'
    )
    if device == "cuda":
        workspace = [
            r.get("accelerator", {}).get("workspace_bytes", 0) / 2**20
            for r in rows
        ]
        print(f"  Median retained GPU workspace: {med(workspace):.1f} MiB")

# ---------- Matched comparisons ----------
print("\n" + "=" * 76)
print("COMPARISONS — second option versus first option")
print("=" * 76)

lookup = {
    (r["case"], r["algorithm"], r["seed"], r["device"]): r
    for r in results
}
comparisons = [
    (f"{case}: CPU → CUDA", case, "cpu", case, "cuda")
    for case in ("baseline", "small", "large", "other")
]
comparisons += [
    ("FP64 → FP32", "baseline", "cuda", "fp32", "cuda"),
    (
        "Large brain: CPU → GPU action prediction",
        "large", "cuda", "large-gpu-inference", "cuda",
    ),
]
for name in ("batch8", "batch32", "workspace32", "workspace64"):
    comparisons.append(
        (f"Baseline → {name}", "baseline", "cuda", name, "cuda")
    )

for label, base_case, base_device, test_case, test_device in comparisons:
    for algorithm in sorted({r["algorithm"] for r in results}):
        pairs = []
        for seed in SEEDS:
            a = lookup.get((base_case, algorithm, seed, base_device))
            b = lookup.get((test_case, algorithm, seed, test_device))
            if a and b:
                pairs.append((a, b))
        if not pairs:
            continue

        matched = all(
            a["steps"] == b["steps"]
            and a["updates"] == b["updates"]
            and a["updates"] > 0
            for a, b in pairs
        )
        print(f"\n{algorithm.upper()} | {label} | {len(pairs)} seed pairs")

        if matched:
            ratio = med([
                a["loop_seconds"] / b["loop_seconds"] for a, b in pairs
            ])
            if ratio >= 1:
                print(
                    f"  {ratio:.2f}× faster; "
                    f"{100*(1-1/ratio):.1f}% less loop time."
                )
            else:
                print(
                    f"  {1/ratio:.2f}× as long; "
                    f"{100*(1/ratio-1):.1f}% more loop time."
                )
        else:
            print("  Step/update counts differ: no clean speedup claim.")

        ram_delta = med([
            (b["peak_rss_bytes"] - a["peak_rss_bytes"]) / 2**20
            for a, b in pairs
        ])
        print(f"  Paired peak host RAM change: {ram_delta:+.1f} MiB")

# ---------- Instrumented breakdown ----------
print("\n" + "=" * 76)
print("PROFILE BREAKDOWN — attribution only, not speedup measurements")
print("=" * 76)

for r in results:
    if not r["profiled"]:
        continue
    print(f'\n{r["algorithm"].upper()} | {r["case"]}')
    print(f'  Full loop: {r["loop_seconds"]:.3f} s')
    a = r.get("accelerator", {})
    for label, key in [
        ("CPU packing", "packing_seconds"),
        ("CPU optimizer/unpacking", "cpu_optimizer_seconds"),
        ("CUDA upload", "cuda_upload_seconds"),
        ("GPU computation", "cuda_compute_seconds"),
        ("CUDA download", "cuda_download_seconds"),
    ]:
        if key in a:
            print(f"  {label}: {a[key]:.3f} s")

print("\nEARLY TERMINATIONS")
early = [r for r in results if r["steps"] < r["requested_steps"]]
if not early:
    print("  None.")
for r in early:
    print(
        f'  {r["case"]}/{r["algorithm"]}/{r["device"]}/seed{r["seed"]}: '
        f'{r["steps"]}/{r["requested_steps"]} steps.'
    )

print(f"\nFAILURES: {len(errors)}")
for label, message in errors:
    print(f"\n{label}\n{message}")

print(f"\nTotal elapsed: {(time.perf_counter()-suite_start)/60:.1f} minutes")
print(
    "\nInterpretation:\n"
    "• Matching counts does not prove identical trajectories.\n"
    "• Different seeds measure different worlds, not timing noise alone.\n"
    "• GPU memory is sampled device-wide usage, including CUDA context.\n"
    "• Sampling adds overhead and can miss brief memory peaks.\n"
    "• Profile counters are not a complete additive breakdown.\n"
    "• Playback/checkpoint I/O is excluded."
)

Repository: /content/wtf
Python: 3.13.16
Wed Nov 25 13:11:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             34W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------